# Self-Supervised (DINOv2) vs. Supervised CNN Representations for Galaxy Morphology Classification
### A rigorous, multi-seed label-efficiency and ablation study

**Course project — Machine Learning**

| At a glance | |
|---|---|
| **Task** | Classify galaxy images into 5 morphology (shape) classes |
| **Data** | Galaxy Zoo 2 (GZ2) images together with volunteer vote fractions |
| **Models compared** | Frozen DINOv2 + MLP probe, frozen DINOv2 + linear probe, fine-tuned DINOv2, ResNet-18 from scratch, ImageNet-pretrained ResNet-18 |
| **Main question** | Does a self-supervised model need fewer labelled galaxies than a supervised CNN to reach the same accuracy? |
| **Metrics** | Accuracy, macro-F1, per-class precision / recall / F1, and computational cost (parameters, time, memory) |
| **Rigour** | 6 label fractions × 5 random seeds, mean ± std, 95% confidence intervals, paired significance tests |
| **Run environment** | Google Colab with a GPU (`Runtime → Change runtime type → T4 / A100 GPU`) |

## How to Read This Notebook

Every step in this notebook follows the same three-part pattern, so that you always know the reason for a step before you see its code:

- **Why?** — the reason the step is needed and the question it helps to answer.
- **What?** — what exactly is produced or decided in the step.
- **How?** — the method used and the main functions involved.

Whenever a technical idea appears, a short **In simple words** note explains it in everyday language. A glossary ("Key Terms in Plain Language") is also given before the steps begin.

**Roadmap of the notebook**

| Step | Title | Purpose in one line |
|---|---|---|
| 1 | Setup and configuration | Prepare libraries, device and experiment settings |
| 2 | Data acquisition | Download the GZ2 images and volunteer votes |
| 3 | Defining morphology classes | Turn vote fractions into 5 well-defined classes |
| 4 | Working dataset and class imbalance | Build a manageable, fair dataset and set class weights |
| 5 | Preprocessing, augmentation and splits | Prepare images and create fixed train / validation / test sets |
| 6 | Models | Define the five models that are compared |
| 7 | Multi-seed, multi-fraction experiments | Train and test every model repeatedly on shrinking label budgets |
| 8 | Aggregated results | Mean ± std and 95% CI; label-efficiency curve |
| 9 | Statistical significance | Check whether differences are real or due to chance |
| 10 | Per-class precision, recall, F1 | See which classes are easy or hard |
| 11 | Confusion matrices | See which classes get mixed up |
| 12 | Error analysis | Summarise the weakest classes and most frequent confusions |
| 13 | t-SNE of embeddings | Visualise how DINOv2 organises galaxies |
| 14 | Attention visualisation | See where DINOv2 "looks" in a galaxy image |
| 15 | Computational efficiency | Compare parameters, time and memory |
| 16 | Ablation studies | Find out which design choice gives the gain |
| 17 | Comparison with AstroDINO | Place our numbers in the context of the source paper |
| 18 | Auto-generated results summary | Produce discussion text from the computed numbers |

## Problem Statement

### Background — why does this matter?
The *shape* of a galaxy (its **morphology**) — whether it is round and smooth, a spiral, or a thin disc seen edge-on — gives important clues about how the galaxy formed and how it has evolved. Deep-learning classifiers can assign these shape classes automatically, but the usual *supervised* approach needs thousands of images that humans have already labelled. Labelling is slow and costly: the Galaxy Zoo project relied on a very large volunteer effort even for a few lakh galaxies. Upcoming surveys such as Euclid and the Vera C. Rubin Observatory's LSST are expected to image billions of galaxies, so labelling everything by hand is simply not possible.

### The problem
**Labelled galaxy images are scarce, while unlabelled images are plentiful.** *Self-supervised* models such as DINOv2 learn useful image features without any labels. The open question is:

> **If we freeze a self-supervised DINOv2 model (which has never seen a galaxy label) and train only a tiny classifier on its features, can it match or beat a supervised CNN trained on the same small set of galaxy labels?**

More formally: given galaxy images *x* and five morphology classes *y* derived from Galaxy Zoo 2 votes, and a labelled-data budget *f* ∈ {1%, 5%, 10%, 25%, 50%, 100%} of the training set, we compare the test performance of five models under **identical** data splits, preprocessing and evaluation rules.

### Research questions

| # | Research question | Answered in |
|---|---|---|
| RQ1 | With all labels available, how does a frozen DINOv2 + MLP probe compare with a supervised ResNet-18? | Steps 8, 9 |
| RQ2 | As labels become scarce (100% → 1%), does the self-supervised model lose less accuracy than the supervised CNN? | Steps 8, 9 |
| RQ3 | Does a non-linear (MLP) probe beat a plain linear probe on the same frozen features? | Steps 9, 16 |
| RQ4 | Does fine-tuning the last transformer blocks of DINOv2 help further? | Step 16 |
| RQ5 | Which morphology classes are easy or hard, and which pairs are confused? | Steps 10 – 12 |
| RQ6 | What does each approach cost in parameters, training time, inference time and GPU memory? | Step 15 |

### Working hypothesis (to be tested, not assumed)
Features learnt without labels should be more *label-efficient*, so the advantage of DINOv2 should be largest when only 1–10% of labels are available. The source paper's own table (Step 17) shows that CNNs trained from scratch can match or beat frozen DINOv2 when *all* labels are available — so the interesting question is whether this order reverses when labels are scarce. The notebook is designed so that the data can also contradict our hypothesis.

### Scope and success criteria
- **In scope:** a 5-class morphology task on a class-balanced subset of GZ2 (up to 20,000 galaxies), the small DINOv2 ViT-S/14 backbone, five models, six label fractions, five seeds.
- **Success is judged by:** test accuracy and macro-F1 (mean ± std over seeds), paired significance tests, per-class behaviour, and computational cost.
- **Out of scope:** other galaxy surveys, larger DINOv2 backbones, and hyperparameter tuning (see *Limitations and Future Work* at the end).

## Objectives and Contributions

**Objectives**

1. Build a scientifically defensible, documented galaxy-morphology classification task from GZ2.
2. Reproduce the frozen-DINOv2 + MLP-probe pipeline of AstroDINO on this task.
3. Compare it against a linear probe (ablation), a fine-tuned DINOv2 (new model), a from-scratch ResNet-18 and an ImageNet-pretrained ResNet-18 — under **identical** splits, preprocessing and evaluation protocol.
4. Measure label efficiency (1–100% of labels) using 5-seed mean ± std and significance testing.
5. Report per-class metrics, confusion matrices and a qualitative error analysis.
6. Report and compare computational cost (parameters, time, memory) alongside accuracy.
7. Compare our absolute numbers with AstroDINO's reported Table 1 accuracies, with proper caveats.

**What is reproduced from the source paper** — Manwadkar & Kapare, *"AstroDINO: Self-Supervised Learning on Astronomical Images"* (Stanford, 2025) [ref. 3]:
- Using a pretrained, **frozen** DINOv2 backbone as a feature extractor for galaxy images.
- Training a small MLP probe (1 hidden layer, 256 units, ReLU) on top of the frozen embeddings.
- Evaluating on Galaxy Zoo morphology labels.
- Visualising DINO attention maps as an unsupervised-segmentation-style qualitative check.

**What is this project's own contribution (not in the source paper)**
- A scientifically motivated, documented class scheme built from GZ2 debiased vote fractions with published reliability thresholds (Willett et al. 2013), instead of an arbitrary label rule.
- A **fine-tuned DINOv2** model (last transformer blocks unfrozen) as an additional comparison point.
- A **linear-probe vs. MLP-probe** ablation on the frozen DINOv2 features.
- A **label-efficiency sweep** at 1%, 5%, 10%, 25%, 50% and 100% of the labelled training data.
- **5 random seeds** per (model, label-fraction) cell, with mean ± std and 95% confidence intervals.
- **Paired significance tests** (paired t-test and Wilcoxon signed-rank test).
- **Per-class precision / recall / F1**, confusion matrices and an error analysis for the two headline models.
- A **computational-efficiency comparison**: parameter counts, training time, inference time and peak GPU memory.
- A results summary that is **generated from the notebook's own computed numbers** (Step 18), not hand-written placeholders.

## Key Terms in Plain Language

| Term | Meaning in simple words |
|---|---|
| **Galaxy morphology** | The visible shape and structure of a galaxy — for example round, spiral or edge-on disc. |
| **Galaxy Zoo 2 (GZ2)** | A citizen-science project in which volunteers answered a series of questions about each galaxy image. |
| **Vote fraction** | The share of volunteers who chose a particular answer. *Debiased* means it has been adjusted so that faint or distant galaxies are not unfairly labelled. |
| **Supervised learning** | Learning from examples that come with human-given labels ("this is a spiral"). |
| **Self-supervised learning (SSL)** | Learning useful patterns from images *without* any labels, by solving a puzzle that the images themselves provide. |
| **DINOv2** | A self-supervised model from Meta AI that learns general-purpose image features from a very large collection of ordinary photographs. |
| **Vision Transformer (ViT-S/14)** | A network design that cuts an image into small square patches (here 14 × 14 pixels) and studies how the patches relate to one another. "S" stands for the small version. |
| **Backbone** | The main part of a network that converts an image into numbers. |
| **Embedding (feature vector)** | The list of numbers (384 for this model) that summarises what the network "sees" in one image. |
| **Frozen** | The weights are locked and are not changed during our training. |
| **Probe (linear / MLP)** | A small classifier trained on top of frozen embeddings. *Linear* = a single layer. *MLP* (multi-layer perceptron) = one extra hidden layer with a non-linear step, so it can learn more complex decision boundaries. |
| **Fine-tuning** | Continuing to train some of the pretrained weights on our own task, usually with a small learning rate. |
| **ResNet-18** | A classic 18-layer convolutional neural network (CNN). *From scratch* = random starting weights. *ImageNet-pretrained* = starts from weights already learnt on the ImageNet photo collection. |
| **Label efficiency** | How well a model performs when only a small share of the training labels is available. |
| **Data augmentation** | Creating slightly changed copies of training images (flips, rotations) so that the model learns to ignore changes that do not matter. |
| **Class imbalance and class weights** | Some classes have far fewer images than others. Class weights make mistakes on rare classes count more during training. |
| **Stratified sampling / split** | Picking images class by class so that every class is represented in the right proportion. |
| **Train / validation / test set** | *Train*: the model learns from it. *Validation*: kept aside for tuning choices. *Test*: used only at the end for an honest evaluation. |
| **Accuracy** | The share of galaxies classified correctly. |
| **Precision, recall, F1** | *Precision*: of the galaxies predicted as class X, how many really are X. *Recall*: of the galaxies that really are X, how many were found. *F1*: one score that combines both. **Macro-F1** is the simple average of F1 over all classes, so rare classes count equally. |
| **Mean ± std, 95% CI** | *Mean*: average over seeds. *Std*: how much the results vary between seeds. *95% CI*: the range in which the true average is expected to lie with 95% confidence. |
| **p-value, paired t-test, Wilcoxon test** | The p-value is the chance of seeing a difference at least this large if the two models were actually equal; below 0.05 suggests a real difference. Both tests compare two models seed by seed; the Wilcoxon test does not assume that the differences follow a bell curve. |
| **Confusion matrix** | A table of true class versus predicted class; the diagonal shows correct predictions. |
| **t-SNE** | A method that squeezes high-dimensional embeddings into a 2-D picture; points that are close together are similar. |
| **Attention map** | A picture showing which parts of the image the transformer pays most attention to. |

## Step 1 — Setup and Configuration

**Why?**
A research experiment must be repeatable: anyone who runs this notebook should obtain the same kind of results. This needs the same libraries, a known computing device (GPU or CPU) and a fixed set of experiment settings. The full experiment is also heavy, so we need a way to test the whole pipeline quickly before the long run.

**What?**
- Install and import the required libraries.
- Choose the computing device.
- Fix the experiment settings: random seeds, label fractions, number of training epochs and dataset size.

**How?**
- `pip install` fetches `galaxy-datasets` (data), `torch`, `torchvision`, `timm` (deep learning) and `scikit-learn`, `scipy`, `statsmodels`, `pandas`, `seaborn` (metrics, statistics and plots).
- `DEVICE` selects the GPU automatically when one is available.
- `QUICK_MODE` switches between two sets of settings:

| Setting | `QUICK_MODE = True` (trial run) | `QUICK_MODE = False` (full protocol) |
|---|---|---|
| Seeds | 2 (0, 1) | 5 (0 – 4) |
| Label fractions | 5%, 25%, 100% | 1%, 5%, 10%, 25%, 50%, 100% |
| CNN epochs | 5 | 15 |
| Fine-tuning epochs | 3 | 8 |
| Probe epochs | 20 | 30 |
| Dataset size | 4,000 galaxies | 20,000 galaxies |

> **In simple words:** A *seed* is a number that fixes the "randomness" of an experiment (which images are picked, how the weights start). Repeating an experiment with 5 different seeds tells us whether a result is stable or just lucky. An *epoch* is one complete pass through the training images.

**Note on compute budget.** The full grid (6 label fractions × 5 seeds × 5 models) is expensive. Frozen-DINOv2 probes are cheap to retrain because the embeddings are computed only once; retraining the CNNs and the fine-tuned DINOv2 is the costly part. A sensible workflow is to run once with `QUICK_MODE = True` to confirm that everything works, and then switch to `False` for the final run.

In [ ]:
!pip install -q galaxy-datasets torch torchvision timm scikit-learn seaborn pandas scipy statsmodels

In [ ]:
import os, time, random, math, json, itertools, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats as sstats

# Use the GPU if Colab has allotted one; otherwise fall back to the (much slower) CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- Compute-budget switch ----
# QUICK_MODE=True : quick trial run (fewer seeds / fractions / epochs) to confirm that the whole
#                   pipeline works from start to end before committing to the full run.
# QUICK_MODE=False: the FULL protocol -- 5 seeds and the complete 1-100% label-efficiency sweep.
QUICK_MODE = False

if QUICK_MODE:
    SEEDS = [0, 1]
    FRACTIONS = [0.05, 0.25, 1.0]
    CNN_EPOCHS = 5
    FT_EPOCHS = 3
    PROBE_EPOCHS = 20
    N_DATASET = 4000
else:
    SEEDS = [0, 1, 2, 3, 4]
    FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
    CNN_EPOCHS = 15
    FT_EPOCHS = 8
    PROBE_EPOCHS = 30
    N_DATASET = 20000  # increase further if Colab disk/RAM permit

print(f"QUICK_MODE={QUICK_MODE} | seeds={SEEDS} | fractions={FRACTIONS} | N_DATASET={N_DATASET}")

## Step 2 — Data Acquisition

**Why?**
We need galaxy images with reliable shape information. Galaxy Zoo 2 is a standard benchmark and is the same underlying label source that the source paper uses, which keeps our study comparable in spirit.

**What?**
The GZ2 *catalogue* (a table with one row per galaxy, holding the image path `file_loc` and the volunteer vote fractions) and the galaxy images themselves.

**How?**
The maintained `galaxy-datasets` package (Walmsley et al.) downloads both. `gz2(root, train=True, download=True)` returns the catalogue as a pandas table and the list of label columns. We print the table's shape and the names of the fraction columns, because Step 3 finds the required columns by their names.

> **In simple words:** Each galaxy was shown to many volunteers who answered questions such as "Is the galaxy smooth or does it have features?". The catalogue stores what share of volunteers gave each answer.

**What to check:** the printed list should contain names ending in `_fraction`. Look at them once by eye before moving on.

In [ ]:
from galaxy_datasets import gz2

DATA_DIR = "/content/gz2_data"
os.makedirs(DATA_DIR, exist_ok=True)

# catalog = table with one row per galaxy; label_cols = names of the volunteer-vote columns.
catalog, label_cols = gz2(root=DATA_DIR, train=True, download=True)
print("Catalogue shape:", catalog.shape)
print("Number of label columns:", len(label_cols))
# Look at the actual column names before relying on the matching logic in Step 3.
# Different galaxy-datasets versions name the columns slightly differently
# (e.g. 'smooth-or-featured-gz2_smooth_fraction' vs 'smooth-or-featured_smooth_fraction'),
# so print the list once and check it by eye.
print([c for c in label_cols if "fraction" in c][:30])

## Step 3 — Scientifically Grounded Morphology Labels *(our contribution)*

**Why?**
Labels are the ground truth for everything that follows. GZ2 does not give one class per galaxy; it gives vote fractions for a tree of questions. If we convert these into classes with an arbitrary rule (for example, an `argmax` over whichever columns load first), the results become hard to defend. We therefore follow the published GZ2 "clean sample" scheme (Willett et al. 2013, *MNRAS* 435:2835; Hart et al. 2016).

**What?**
Five well-separated, astrophysically meaningful classes, each defined by thresholds on the GZ2 debiased vote fractions, plus a minimum-vote-count reliability cut:

| Class | Definition (GZ2 debiased vote fractions) |
|---|---|
| 0 — Round smooth (elliptical) | smooth branch, `p_round ≥ 0.5` |
| 1 — In-between smooth | smooth branch, `p_in_between ≥ 0.5` |
| 2 — Cigar-shaped smooth | smooth branch, `p_cigar ≥ 0.5` |
| 3 — Edge-on disk | featured / disk branch, `p_edgeon ≥ 0.715` |
| 4 — Spiral (face-on) | featured / disk branch, not edge-on, `p_spiral ≥ 0.619` |

Galaxies that meet none of these thresholds with adequate votes (ambiguous or low-confidence morphology) are **dropped**. This is standard practice for building "clean" GZ2-derived datasets — a deliberate, documented filtering step, not a missing-data problem.

**How?**
1. Find the required column names automatically by matching substrings (Step 3a) rather than hard-coding column positions.
2. Apply a minimum-vote-count cut (N ≥ 10) where a vote-count column exists.
3. Walk down the GZ2 decision tree for every galaxy (Step 3b), as shown below, and print the class sizes as a sanity check.

```text
Smooth or featured?
├── Smooth (fraction ≥ 0.5)  →  best of: Round | In-between | Cigar-shaped   (each needs ≥ 0.5)
└── Featured / disk          →  Edge-on? (≥ 0.715)  →  Class 3: Edge-on disk
                                 else Spiral arms? (≥ 0.619)  →  Class 4: Spiral
                                 else → dropped as ambiguous
```

> **In simple words:** We copy the way volunteers were asked the questions. First ask "smooth or featured?", and only then ask the follow-up question that belongs to that branch. Galaxies on which volunteers did not agree clearly are left out, so that the labels we train on are trustworthy.

**Design note — why the order of the checks matters.** The `round`, `in-between` and `cigar` columns are *conditional* vote fractions: they mean something only for galaxies that volunteers first called "smooth". An earlier version of this notebook checked them for every galaxy without first looking at the smooth-vs-featured vote. That wrongly pushed about 97.6% of galaxies into a smooth class and left only 62 edge-on disks out of 1,67,114 galaxies. The corrected logic applies the top-level smooth-vs-featured gate first, exactly as the GZ2 decision tree does, and only then looks inside the relevant branch. Edge-on disks are still genuinely rarer than round or spiral galaxies, so the remaining imbalance is handled by class-weighted loss (Step 4) and stratified subsampling (Step 7).

**Check before you run.** Based on the package's label metadata, the resolved columns should look like `smooth-or-featured-gz2_smooth_fraction`, `how-rounded-gz2_round_fraction`, `how-rounded-gz2_in-between_fraction`, `how-rounded-gz2_cigar_fraction`, `disk-edge-on-gz2_yes_fraction` and `has-spiral-arms-gz2_yes_fraction` (exact spelling can vary between package versions). Because the matching works on substrings, a column such as `bulge-shape-gz2_round_fraction`, which also contains "round", could be picked by mistake if it appears earlier in the table. Compare the printed "Resolved columns" with the list above before trusting the class counts.

### Step 3a — Locate the required columns
**Why?** Column names differ slightly between package versions. **What?** The seven columns needed for labelling (smooth, round, in-between, cigar, edge-on, spiral and, optionally, a vote count). **How?** `find_col()` returns the first column whose name contains all the given substrings; the cell then stops with a clear error message if any required column is missing, and applies the vote-count cut if a count column exists.

In [ ]:
def find_col(candidates_substrings, cols):
    # Return the first column whose name contains ALL the given substrings (upper/lower case ignored).
    for c in cols:
        cl = c.lower()
        if all(s in cl for s in candidates_substrings):
            return c
    return None

cols = list(catalog.columns)

col_smooth      = find_col(["smooth-or-featured", "smooth", "fraction"], cols)
col_featured    = find_col(["smooth-or-featured", "featured", "fraction"], cols) or find_col(["disk", "fraction"], cols)
col_round       = find_col(["round", "fraction"], cols)
col_inbetween   = find_col(["in-between", "fraction"], cols) or find_col(["inbetween", "fraction"], cols)
col_cigar       = find_col(["cigar", "fraction"], cols)
col_edgeon      = find_col(["edge-on", "yes", "fraction"], cols) or find_col(["edgeon", "yes", "fraction"], cols)
col_spiral      = find_col(["spiral-arms", "yes", "fraction"], cols) or find_col(["has-spiral", "yes", "fraction"], cols)

# Vote-count column for a minimum-reliability cut (in the style of Willett et al. 2013).
# Not every galaxy-datasets version stores this under the same name, so it is optional:
# if it is not found, we skip the count-based cut and rely on the fraction thresholds alone.
col_smooth_count = (find_col(["smooth-or-featured", "count"], cols)
                     or find_col(["smooth-or-featured", "total-votes"], cols)
                     or find_col(["smooth-or-featured", "total_votes"], cols))

found = {
    "smooth": col_smooth, "round": col_round, "in_between": col_inbetween, "cigar": col_cigar,
    "edgeon": col_edgeon, "spiral": col_spiral, "smooth_count (optional)": col_smooth_count,
}
print("Resolved columns:", found)
assert all(found[k] is not None for k in ["smooth", "round", "in_between", "cigar", "edgeon", "spiral"]), (
    "One or more required vote-fraction columns could not be resolved automatically -- run print(cols) "
    "and adjust the substrings in find_col() to match your galaxy-datasets version before proceeding."
)

MIN_VOTES = 10
if col_smooth_count is not None:
    before = len(catalog)
    catalog = catalog[catalog[col_smooth_count] >= MIN_VOTES].copy()
    print(f"Applied minimum-vote-count reliability cut (N>={MIN_VOTES} on {col_smooth_count}): "
          f"{len(catalog)} / {before} galaxies retained.")
else:
    print("No vote-count column resolved for this schema version -- skipping the count-based "
          "reliability cut; the fraction thresholds in Step 3 still apply.")

### Step 3b — Assign one class to each galaxy
**Why?** The models need a single class label per image. **What?** A `label` column with values 0 – 4, and the "clean" table with ambiguous galaxies removed. **How?** `assign_gz2_class()` applies the decision-tree logic described above row by row; galaxies that get `-1` are dropped. The printed class counts and the max / min class ratio tell us how imbalanced the clean sample is.

In [ ]:
# DESIGN NOTE: the round / in-between / cigar columns are *conditional* debiased vote fractions --
# they mean something only for galaxies that volunteers first called "smooth". So we check the
# top-level smooth-vs-featured fraction FIRST (exactly like the GZ2 decision tree) and only then
# look inside the relevant branch. An earlier version skipped this gate; it pushed ~97.6% of all
# galaxies into a smooth class and left only 62 edge-on disks out of 1,67,114 galaxies.

SMOOTH_GATE = 0.5      # top-level question: smooth vs. featured/disk
SUBCLASS_MIN = 0.5     # within-branch confidence floor

def assign_gz2_class(row):
    if row[col_smooth] >= SMOOTH_GATE:
        # Smooth branch: take the best-supported of round / in-between / cigar.
        smooth_opts = {0: row[col_round], 1: row[col_inbetween], 2: row[col_cigar]}
        best_class, best_frac = max(smooth_opts.items(), key=lambda kv: kv[1])
        return best_class if best_frac >= SUBCLASS_MIN else -1
    else:
        # Featured/disk branch.
        if row[col_edgeon] >= 0.715:
            return 3  # edge-on disk
        if row[col_spiral] >= 0.619:
            return 4  # face-on spiral
        return -1  # ambiguous within the featured branch -> dropped

CLASS_NAMES = ["Round smooth", "In-between smooth", "Cigar-shaped smooth", "Edge-on disk", "Spiral"]
N_CLASSES = len(CLASS_NAMES)

catalog["label"] = catalog.apply(assign_gz2_class, axis=1)
clean = catalog[catalog["label"] >= 0].copy()

print(f"Retained {len(clean)} / {len(catalog)} galaxies after clean-sample filtering "
      f"({100*len(clean)/len(catalog):.1f}%).")
print("\nClass distribution (clean sample, after gating fix):")
dist = clean["label"].value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES)))
print(dist)
imbalance_ratio = dist.max() / dist.min()
print(f"\nMax/min class-count ratio: {imbalance_ratio:.1f}x")
if imbalance_ratio > 10:
    print("Still notably imbalanced -- Step 4 applies class-weighted loss during training as "
          "a second safeguard on top of this fix, and Step 7 uses stratified (per-class) "
          "subsampling for the label-efficiency sweep so that every fraction retains all 5 classes.")

## Step 4 — Working Dataset and Class Imbalance

**Why?**
- Training 5 models × 6 label fractions × 5 seeds on the whole GZ2 catalogue (well over a lakh of galaxies) is too heavy for a Colab session, so a smaller working dataset is needed.
- If one class has far fewer images than the others, a model can score a high accuracy by mostly predicting the big classes and ignoring the rare one.

**What?**
- A working dataset of up to `N_DATASET` galaxies (20,000 in the full run) with an equal cap of `N_DATASET / 5` galaxies per class. A class with fewer galaxies than the cap keeps all of them.
- A vector of *class weights* used in the loss function of every model.

**How?**
- `groupby("label")` followed by `sample(...)` draws at most `per_class_cap` galaxies from each class (fixed `random_state`, so the subset is the same every time).
- `compute_class_weight("balanced")` gives each class the weight *N ÷ (K × nᶜ)*, where *N* is the total number of galaxies, *K* the number of classes and *nᶜ* the size of class *c*. Rare classes therefore get larger weights, and these weights are passed to `CrossEntropyLoss` for every model.

> **In simple words:** Class weights are like giving extra marks for getting a rare class right (and extra penalty for missing it), so that the model does not simply "play safe" by always guessing the common classes.

**What to check:** the bar chart and the printed table should show all five classes. If the smallest class is still very small, read its results with extra caution.

In [ ]:
# Cap the dataset size (equal number of galaxies per class, wherever available) so that training
# stays practical on Colab. Increase N_DATASET in Step 1 if your compute budget allows a larger run.
per_class_cap = max(1, N_DATASET // N_CLASSES)
sub = clean.groupby("label", group_keys=False).apply(
    lambda g: g.sample(min(len(g), per_class_cap), random_state=0)
).reset_index(drop=True)

print("Final dataset size:", len(sub))
dist2 = sub["label"].value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES)))
print(dist2)

fig, ax = plt.subplots(figsize=(6, 4))
dist2.plot(kind="bar", ax=ax)
ax.set_ylabel("Count"); ax.set_title("Class distribution used for training/evaluation (after gating fix)")
plt.tight_layout(); plt.savefig("class_distribution.png", dpi=150); plt.show()

# Second safeguard for any leftover imbalance: class-weighted loss. Even after fixing the gating,
# the real GZ2 population has fewer edge-on disks than round or spiral galaxies. We therefore
# compute "balanced" (inverse-frequency) class weights and pass them to every CrossEntropyLoss
# below, so that mistakes on rarer classes cost more and the model cannot get away with simply
# predicting the majority class.
class_weights_np = compute_class_weight(class_weight="balanced", classes=np.arange(N_CLASSES), y=sub["label"].values)
CLASS_WEIGHTS = torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)
print("\nBalanced class weights (used in CrossEntropyLoss for every model below):")
for name, w in zip(CLASS_NAMES, class_weights_np):
    print(f"  {name}: {w:.3f}")

min_class_count = dist2.min()
if min_class_count < 50:
    print(f"\nNote: the smallest class still has only {min_class_count} examples after capping. "
          f"Class-weighted loss (above) and stratified per-class label-fraction sampling (Step 7) "
          f"reduce this problem, but results for this class should be read with wider uncertainty in mind.")

## Step 5 — Preprocessing, Augmentation and Data Splits

**Why?**
Pretrained networks expect images of a fixed size and brightness scale. Augmentation helps the models to generalise, and — most importantly for a fair comparison — every model in every experiment must be tested on exactly the same unseen images.

**What?**
Image transforms, a fixed 70 / 15 / 15 train / validation / test split, and a small `Dataset` class that reads an image and its label.

**How?**
- **Resize:** all images to 224 × 224 (the standard input size for DINOv2 and ResNet-18).
- **Normalisation:** ImageNet channel mean and standard deviation, matching what the pretrained backbones expect.
- **Training-time augmentation:** random horizontal flip (p = 0.5) and random rotation (±15°). The orientation of a galaxy on the sky is arbitrary, so flips and rotations do not change its class and are standard for this domain. **No colour jitter** is used, because colour carries real physical information (stellar population, redshift proxies) that we do not want to disturb.
- **Split:** stratified 70 / 15 / 15 by class, **fixed across all models and experiments** (same `random_state`), so every model in every comparison in Steps 8 – 16 sees exactly the same test set.
- **Label-fraction subsampling (Step 7):** done only *inside* the fixed training split, so the validation and test sets never change or shrink across fractions.
- **Validation set:** it is reserved for future tuning. In this notebook all models use fixed learning rates and epoch counts, so the test set is touched only for the final evaluation.

> **In simple words:** *Normalisation* rescales pixel values to the range the pretrained model was trained on. *Augmentation* shows the model rotated and mirrored versions of the same galaxy so that it does not learn to depend on the sky angle.

In [ ]:
# Stratified 70/15/15 split. The same random_state is used everywhere, so every model sees
# exactly the same train / validation / test images.
train_df, temp_df = train_test_split(sub, test_size=0.30, stratify=sub["label"], random_state=0)
val_df, test_df    = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=0)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Reads one image from disk, applies the chosen transform and returns (image tensor, class label).
class GalaxyImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.transform(Image.open(row["file_loc"]).convert("RGB"))
        return img, int(row["label"])

train_ds_aug   = GalaxyImageDataset(train_df, train_transform)
train_ds_plain = GalaxyImageDataset(train_df, eval_transform)
val_ds         = GalaxyImageDataset(val_df, eval_transform)
test_ds        = GalaxyImageDataset(test_df, eval_transform)

## Step 6 — Models

**Why?**
Each research question in the problem statement needs a specific model to answer it. All models must be trained and tested in the same way, so that any difference in results comes from the model and not from the setup.

**What?** Five models:

| # | Model | What is trained | Purpose |
|---|---|---|---|
| 1 | **DINOv2 (frozen) + MLP probe** | Small MLP only | Reproduction of the source paper's pipeline |
| 2 | **DINOv2 (frozen) + linear probe** | One linear layer only | Ablation: does the non-linearity of the MLP matter? |
| 3 | **ResNet-18 (from scratch)** | Whole network, random start | Supervised baseline without pretraining |
| 4 | **ResNet-18 (ImageNet fine-tuned)** | Whole network, ImageNet start | Supervised baseline with transfer learning |
| 5 | **DINOv2 (fine-tuned)** | Last 2 transformer blocks + linear head | Our extension: does adapting the backbone help? |

The numbering above is the same as in the experiment loops of Step 7.

**How?**
Sub-steps 6a – 6e below load the DINOv2 backbone, compute its embeddings once, and define the training helpers for the probes, the ResNet-18 models and the fine-tuned DINOv2.

### Step 6a — Load the DINOv2 backbone
**Why?** It is the pretrained feature extractor for models 1, 2 and 5. **What?** DINOv2 ViT-S/14 (the small version), which turns each image into a 384-number embedding. **How?** `torch.hub.load` fetches it from Meta AI's repository; `freeze()` switches off gradient updates; `count_params()` counts (trainable) parameters for the efficiency table in Step 15.

In [ ]:
# Load the pretrained DINOv2 ViT-S/14 backbone (weights are fetched from Meta AI via torch.hub).
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dinov2_backbone.eval().to(DEVICE)
EMBED_DIM = dinov2_backbone.embed_dim
print("DINOv2 embed dim:", EMBED_DIM)

# Freezing = switching off gradient updates, so the weights stay exactly as pretrained.
def freeze(model):
    for p in model.parameters():
        p.requires_grad = False

def count_params(model, trainable_only=True):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

### Step 6b — Extract embeddings once
**Why?** With a frozen backbone the embeddings never change, so computing them once saves a great deal of time: every probe, for every fraction and seed, can then be trained in seconds. **What?** Embeddings and labels for the train, validation and test sets. **How?** `extract_embeddings()` passes each image through the backbone in batches with gradients switched off; the total time is stored in `dino_extract_time`. The training embeddings use the plain (non-augmented) images.

> **In simple words:** The backbone acts like a fixed "translator" that converts every galaxy picture into a short list of numbers. The probes only learn to classify those lists.

In [ ]:
# Pass every image through the frozen backbone ONCE and keep the resulting embeddings
# (one list of numbers per image); the probes are later trained on these stored numbers.
@torch.no_grad()
def extract_embeddings(backbone, dataset, batch_size=64):
    backbone.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    for imgs, y in loader:
        out = backbone(imgs.to(DEVICE))
        feats.append(out.cpu().numpy()); labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

freeze(dinov2_backbone)
t0 = time.time()
train_feats_full, train_labels_full = extract_embeddings(dinov2_backbone, train_ds_plain)
val_feats,  val_labels  = extract_embeddings(dinov2_backbone, val_ds)
test_feats, test_labels = extract_embeddings(dinov2_backbone, test_ds)
dino_extract_time = time.time() - t0
print("Frozen-DINOv2 embedding extraction time (s):", round(dino_extract_time, 1))

### Step 6c — Probes (small classifiers on frozen embeddings)
**Why?** To test the source paper's MLP probe (model 1) and a simpler linear probe (model 2). **What?** Two tiny networks: `MLPProbe` (384 → 256 → ReLU → 5) and `LinearProbe` (384 → 5), plus a training function. **How?** `train_probe()` trains with the Adam optimiser and class-weighted cross-entropy loss, fixes the seed for repeatability, and returns predictions together with training and inference times.

In [ ]:
# Probe 1 (MLP): one hidden layer of 256 units with a ReLU non-linearity, as in the AstroDINO paper.
class MLPProbe(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x): return self.net(x)

# Probe 2 (linear): a single layer with no non-linearity -- used for the ablation.
class LinearProbe(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Linear(in_dim, n_classes)
    def forward(self, x): return self.net(x)

# Train a probe on pre-computed embeddings (Adam optimiser, class-weighted cross-entropy loss).
# Returns the trained model, test predictions, training time and inference time per image.
def train_probe(probe_cls, X_train, y_train, X_eval, y_eval, n_classes, epochs, batch_size=64, lr=1e-3,
                 seed=0, class_weights=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = probe_cls(X_train.shape[1], n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    ds = torch.utils.data.TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                         torch.tensor(y_train, dtype=torch.long))
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, generator=g)

    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    train_time = time.time() - t0

    model.eval()
    t0 = time.time()
    with torch.no_grad():
        preds = model(torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)).argmax(1).cpu().numpy()
    infer_time_per_img = (time.time() - t0) / max(1, len(X_eval))
    return model, preds, train_time, infer_time_per_img

### Step 6d — ResNet-18 baselines
**Why?** They represent the supervised alternative (models 3 and 4). **What?** A ResNet-18 whose final layer is replaced to give 5 outputs; `pretrained=False` starts from random weights, `pretrained=True` starts from ImageNet weights. **How?** `train_cnn()` trains the whole network end to end (Adam, learning rate 1e-4, class-weighted loss) and records training time, inference time per image and peak GPU memory.

In [ ]:
# ResNet-18 whose last layer is replaced to output 5 classes.
# pretrained=True starts from ImageNet weights; pretrained=False starts from random weights.
def build_resnet18(n_classes, pretrained):
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model.to(DEVICE)

# Train the whole CNN end to end and record training time, inference time and peak GPU memory.
def train_cnn(model, train_dataset, eval_dataset, epochs, batch_size=64, lr=1e-4, seed=0, class_weights=None):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, generator=g)
    eval_loader  = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    train_time = time.time() - t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0

    model.eval()
    all_preds, all_labels = [], []
    t0 = time.time()
    with torch.no_grad():
        for xb, yb in eval_loader:
            preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.append(preds); all_labels.append(yb.numpy())
    infer_time_per_img = (time.time() - t0) / max(1, len(eval_dataset))
    return model, np.concatenate(all_preds), np.concatenate(all_labels), train_time, infer_time_per_img, peak_mem_mb

### Step 6e — Fine-tuned DINOv2
**Why?** To find out whether adapting the backbone to galaxies improves on the frozen features (model 5). **What?** DINOv2 in which only the last `n_unfrozen_blocks = 2` transformer blocks and a new linear head are trainable. **How?** `build_dinov2_finetune_model()` freezes everything and then unfreezes the last blocks; `train_dinov2_finetune()` trains them with a very small learning rate (1e-5), so that the useful pretrained knowledge is not destroyed.

> **In simple words:** *Fine-tuning* is like letting a trained expert adjust a little to a new field, instead of teaching a beginner from zero.

In [ ]:
# DINOv2 in which only the last few transformer blocks (plus a new linear head) are trainable.
def build_dinov2_finetune_model(n_classes, n_unfrozen_blocks=2):
    backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    freeze(backbone)
    for blk in backbone.blocks[-n_unfrozen_blocks:]:
        for p in blk.parameters():
            p.requires_grad = True

    class DinoV2Classifier(nn.Module):
        def __init__(self, backbone, embed_dim, n_classes):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Linear(embed_dim, n_classes)
        def forward(self, x):
            feats = self.backbone(x)
            return self.head(feats)

    model = DinoV2Classifier(backbone, backbone.embed_dim, n_classes).to(DEVICE)
    return model

# Fine-tune with a very small learning rate so that the pretrained knowledge is not destroyed.
def train_dinov2_finetune(model, train_dataset, eval_dataset, epochs, batch_size=32, lr=1e-5, seed=0,
                           class_weights=None):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(trainable, lr=lr)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, generator=g)
    eval_loader  = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    train_time = time.time() - t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0

    model.eval()
    all_preds, all_labels = [], []
    t0 = time.time()
    with torch.no_grad():
        for xb, yb in eval_loader:
            preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.append(preds); all_labels.append(yb.numpy())
    infer_time_per_img = (time.time() - t0) / max(1, len(eval_dataset))
    return model, np.concatenate(all_preds), np.concatenate(all_labels), train_time, infer_time_per_img, peak_mem_mb

## Step 7 — Unified Multi-Seed, Multi-Fraction Experiment Loop

**Why?**
A single training run can be lucky or unlucky. To make trustworthy claims we need repeated runs (several seeds), and to study label efficiency we need several label budgets (fractions).

**What?**
A table of records with one row for every (model, fraction, seed) run, holding the predictions on the test set, the number of labelled images used, training and inference times, parameter counts and (for GPU models) peak memory.

**How?**
Every model is retrained from scratch for each (fraction, seed) pair. All runs draw from the same fixed training split and are evaluated on the same fixed test set. For a given (fraction, seed), **all models receive the same labelled subset**, which is what makes the seed-by-seed (paired) tests in Step 9 valid.

### Step 7a — Stratified subsampling helper
**Why?** With plain random sampling, a small class such as Edge-on disk could receive *zero* examples at 1% labels, silently turning the experiment into a 4-class problem. **What?** A function that returns the indices of a labelled subset. **How?** `stratified_subsample()` samples the chosen fraction *within each class* (at least one image per class), using a seed derived from the run's seed and fraction.

In [ ]:
def stratified_subsample(labels_array, frac, seed, n_classes):
    # Stratified (per-class) subsampling for the label-efficiency sweep. Plain random sampling of a
    # tiny class such as Edge-on disk could easily pick ZERO examples of it at frac=0.01, quietly
    # turning that experiment into a 4-class problem. Sampling a fraction from WITHIN each class
    # guarantees that every class appears (at least 1 example) at every label fraction, which keeps
    # the label-efficiency curve comparable across fractions and consistent with the class-weighted loss.
    rng = np.random.RandomState(seed)
    sub_idx = []
    for cls in range(n_classes):
        cls_idx = np.where(labels_array == cls)[0]
        if len(cls_idx) == 0:
            continue
        n_c = len(cls_idx) if frac >= 1.0 else max(1, int(round(len(cls_idx) * frac)))
        n_c = min(n_c, len(cls_idx))
        sub_idx.append(rng.choice(cls_idx, size=n_c, replace=False))
    sub_idx = np.concatenate(sub_idx)
    rng.shuffle(sub_idx)
    return sub_idx

### Step 7b — Models 1 – 3 at every fraction
**Why?** These give the complete label-efficiency curve. **What?** Frozen DINOv2 + MLP, frozen DINOv2 + linear, and ResNet-18 from scratch, for all fractions × all seeds (in the full run: 6 × 5 = 30 runs per model). **How?** The two probes reuse the pre-computed embeddings, so they are quick; the from-scratch ResNet-18 is the slow one in this loop because the whole network is trained each time, with augmentation.

In [ ]:
n_train_full = len(train_df)

records = []  # one row per (model, fraction, seed)

# For every (fraction, seed) pair: draw a stratified labelled subset, then train and test each model on it.
for frac in FRACTIONS:
    for seed in SEEDS:
        sub_idx = stratified_subsample(train_labels_full, frac, seed=seed * 1000 + int(frac * 100), n_classes=N_CLASSES)
        n_sub = len(sub_idx)

        # --- 1. DINOv2 frozen + MLP probe ---
        _, preds, ttime, itime = train_probe(MLPProbe, train_feats_full[sub_idx], train_labels_full[sub_idx],
                                              test_feats, test_labels, N_CLASSES, PROBE_EPOCHS, seed=seed,
                                              class_weights=CLASS_WEIGHTS)
        records.append(dict(model="DINOv2-frozen+MLP", fraction=frac, seed=seed, n_labeled=n_sub,
                             preds=preds, y_true=test_labels, train_time=ttime, infer_time=itime,
                             n_params=count_params(MLPProbe(EMBED_DIM, N_CLASSES))))

        # --- 2. DINOv2 frozen + linear probe (ablation) ---
        _, preds, ttime, itime = train_probe(LinearProbe, train_feats_full[sub_idx], train_labels_full[sub_idx],
                                              test_feats, test_labels, N_CLASSES, PROBE_EPOCHS, seed=seed,
                                              class_weights=CLASS_WEIGHTS)
        records.append(dict(model="DINOv2-frozen+Linear", fraction=frac, seed=seed, n_labeled=n_sub,
                             preds=preds, y_true=test_labels, train_time=ttime, infer_time=itime,
                             n_params=count_params(LinearProbe(EMBED_DIM, N_CLASSES))))

        # --- 3. ResNet-18 from scratch ---
        cnn = build_resnet18(N_CLASSES, pretrained=False)
        cnn_sub_ds = Subset(train_ds_aug, sub_idx.tolist())
        _, preds, y_t, ttime, itime, mem = train_cnn(cnn, cnn_sub_ds, test_ds, CNN_EPOCHS, seed=seed,
                                                       class_weights=CLASS_WEIGHTS)
        records.append(dict(model="ResNet18-scratch", fraction=frac, seed=seed, n_labeled=n_sub,
                             preds=preds, y_true=y_t, train_time=ttime, infer_time=itime,
                             peak_mem_mb=mem, n_params=count_params(cnn)))

        print(f"[frac={frac:.2f} seed={seed}] n_labeled={n_sub} done: MLP/Linear/CNN-scratch")

print("Models 1-3 done for all fractions and seeds; records so far:", len(records))

### Step 7c — Expensive models at a reduced set of fractions
**Why?** ResNet-18 (ImageNet fine-tuned) and the fine-tuned DINOv2 train the whole network or large parts of it, so running them on the full grid would not fit a single Colab GPU session. **What?** Models 4 and 5 for **every seed but only three fractions**: the smallest, the middle and 100% (in the full run: 1%, 25% and 100%). **How?** `EXPENSIVE_FRACTIONS` picks these three values. This is a deliberate, documented compute trade-off; widen it to the full `FRACTIONS` list if you have the GPU budget.

In [ ]:
# Models 4 and 5 (whole-network training) run only at the smallest, middle and full fractions.
EXPENSIVE_FRACTIONS = sorted(set([FRACTIONS[0], FRACTIONS[len(FRACTIONS)//2], FRACTIONS[-1]]))
print("Fractions used for expensive models:", EXPENSIVE_FRACTIONS)

for frac in EXPENSIVE_FRACTIONS:
    for seed in SEEDS:
        sub_idx = stratified_subsample(train_labels_full, frac, seed=seed * 1000 + int(frac * 100), n_classes=N_CLASSES)
        n_sub = len(sub_idx)

        # --- 4. ResNet-18 ImageNet-finetuned ---
        cnn_ft = build_resnet18(N_CLASSES, pretrained=True)
        cnn_sub_ds = Subset(train_ds_aug, sub_idx.tolist())
        _, preds, y_t, ttime, itime, mem = train_cnn(cnn_ft, cnn_sub_ds, test_ds, CNN_EPOCHS, seed=seed,
                                                       class_weights=CLASS_WEIGHTS)
        records.append(dict(model="ResNet18-ImageNet-FT", fraction=frac, seed=seed, n_labeled=n_sub,
                             preds=preds, y_true=y_t, train_time=ttime, infer_time=itime,
                             peak_mem_mb=mem, n_params=count_params(cnn_ft)))

        # --- 5. DINOv2 fine-tuned (last 2 blocks unfrozen) ---
        dino_ft_model = build_dinov2_finetune_model(N_CLASSES, n_unfrozen_blocks=2)
        dino_sub_ds = Subset(train_ds_aug, sub_idx.tolist())
        _, preds, y_t, ttime, itime, mem = train_dinov2_finetune(dino_ft_model, dino_sub_ds, test_ds,
                                                                   FT_EPOCHS, seed=seed,
                                                                   class_weights=CLASS_WEIGHTS)
        records.append(dict(model="DINOv2-finetuned", fraction=frac, seed=seed, n_labeled=n_sub,
                             preds=preds, y_true=y_t, train_time=ttime, infer_time=itime, peak_mem_mb=mem,
                             n_params=count_params(dino_ft_model, trainable_only=True)))

        print(f"[expensive frac={frac:.2f} seed={seed}] n_labeled={n_sub} done: ResNet18-FT / DINOv2-finetuned")

print("Total records:", len(records))

## Step 8 — Aggregated Results: Mean ± Std and 95% Confidence Intervals

**Why?**
A single number per run is not enough. We need to know the typical performance of each model and how much it varies with the random seed.

**What?**
- `per_run_df`: accuracy, macro-F1 and cost measurements for every single run.
- `agg`: for each (model, fraction), the mean, standard deviation and 95% confidence interval of accuracy and macro-F1.
- The **label-efficiency curve**: test accuracy against the fraction of labelled data, for every model.

**How?**
`summarize()` computes accuracy and macro-F1 from the stored predictions; `groupby(["model", "fraction"])` aggregates over seeds; `plt.errorbar` draws the mean with its 95% CI as error bars.

> **In simple words:** A model whose curve stays high even at small label fractions is *label-efficient* — it learns well from few examples. Error bars show how much the result moves from seed to seed; if two curves' bars overlap a lot, the difference may not be real (Step 9 checks this formally).

**Note on the confidence interval.** The interval uses the normal approximation, 1.96 × standard error. With only 5 seeds, a t-distribution multiplier (about 2.78) would give wider, more conservative intervals, so please treat the plotted intervals as approximate.

In [ ]:
# One row per single run: accuracy and macro-F1 on the fixed test set, plus the cost measurements.
def summarize(records):
    rows = []
    for r in records:
        acc = accuracy_score(r["y_true"], r["preds"])
        f1m = f1_score(r["y_true"], r["preds"], average="macro")
        rows.append(dict(model=r["model"], fraction=r["fraction"], seed=r["seed"],
                          n_labeled=r["n_labeled"], accuracy=acc, macro_f1=f1m,
                          train_time=r["train_time"], infer_time=r["infer_time"],
                          n_params=r["n_params"], peak_mem_mb=r.get("peak_mem_mb", np.nan)))
    return pd.DataFrame(rows)

per_run_df = summarize(records)

# Half-width of the 95% confidence interval (normal approximation: 1.96 x standard error).
def ci95(x):
    x = np.asarray(x)
    if len(x) < 2:
        return 0.0
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x))

agg = per_run_df.groupby(["model", "fraction"]).agg(
    acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std"), acc_ci95=("accuracy", ci95),
    f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"), f1_ci95=("macro_f1", ci95),
    n_runs=("seed", "count"),
).reset_index()
agg

In [ ]:
plt.figure(figsize=(8, 6))
for m, g in agg.groupby("model"):
    g = g.sort_values("fraction")
    plt.errorbar(g["fraction"], g["acc_mean"], yerr=g["acc_ci95"], marker="o", capsize=3, label=m)
plt.xlabel("Fraction of labelled training data")
plt.ylabel("Test accuracy (mean ± 95% CI)")
plt.title("Label efficiency across models (multi-seed)")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("label_efficiency_curve.png", dpi=150)
plt.show()

## Step 9 — Statistical Significance Testing

**Why?**
Two average accuracies can differ simply by chance. A formal test tells us whether an observed gap is larger than what random variation across seeds would normally produce.

**What?**
For each label fraction where two models were run on all seeds, a paired t-test and a Wilcoxon signed-rank test on the per-seed accuracies, for two comparisons: **DINOv2-frozen+MLP vs. ResNet18-scratch** and **DINOv2-frozen+MLP vs. DINOv2-frozen+Linear**.

**How?**
`paired_tests()` lines up the two models' accuracies seed by seed (both saw the same labelled subset), and computes `ttest_rel` (assumes roughly bell-shaped differences) and `wilcoxon` (does not assume this). The column `significant_p<0.05` is based on the paired t-test.

> **In simple words:** The p-value answers "if the two models were really equally good, how surprising would this gap be?" A p-value below 0.05 is taken as evidence of a real difference.

**Points to keep in mind**
- With only 5 seeds, the smallest possible two-sided Wilcoxon p-value is 0.0625, so that test can never fall below 0.05 in the full run. This is why the significance flag uses the t-test; the Wilcoxon result is reported as a robustness check.
- The seeds are repeated trials on one fixed test set. The tests therefore speak about randomness in training and sampling, not about how the result would carry over to other galaxy datasets.

In [ ]:
# Paired tests: for each seed both models were trained on the SAME labelled subset,
# so their accuracies are compared seed by seed.
def paired_tests(df, model_a, model_b, fraction):
    a = df[(df.model == model_a) & (df.fraction == fraction)].sort_values("seed")["accuracy"].values
    b = df[(df.model == model_b) & (df.fraction == fraction)].sort_values("seed")["accuracy"].values
    if len(a) != len(b) or len(a) < 2:
        return None
    t_stat, t_p = sstats.ttest_rel(a, b)
    try:
        w_stat, w_p = sstats.wilcoxon(a, b)
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    return dict(model_a=model_a, model_b=model_b, fraction=fraction,
                mean_a=a.mean(), mean_b=b.mean(), diff=a.mean() - b.mean(),
                t_stat=t_stat, t_p=t_p, wilcoxon_stat=w_stat, wilcoxon_p=w_p)

comparisons = [("DINOv2-frozen+MLP", "ResNet18-scratch"),
               ("DINOv2-frozen+MLP", "DINOv2-frozen+Linear")]
sig_rows = []
for ma, mb in comparisons:
    for frac in FRACTIONS:
        res = paired_tests(per_run_df, ma, mb, frac)
        if res is not None:
            sig_rows.append(res)
sig_df = pd.DataFrame(sig_rows)
sig_df["significant_p<0.05"] = sig_df["t_p"] < 0.05
sig_df

## Step 10 — Per-Class Precision, Recall and F1

**Why?**
Overall accuracy can hide poor performance on a rare or difficult class. Per-class scores show exactly where a model succeeds or fails.

**What?**
A table of precision, recall and F1 for each of the five classes, for the two headline models (frozen DINOv2 + MLP and ResNet-18 from scratch), using one representative run: 100% labels and the first seed.

**How?**
`per_class_table()` calls scikit-learn's `precision_score`, `recall_score` and `f1_score` with `average=None`, which returns one value per class.

> **In simple words:** *Precision* asks "when the model says Spiral, how often is it right?". *Recall* asks "of all real spirals, how many did the model find?". *F1* balances the two in a single number.

In [ ]:
def per_class_table(y_true, y_pred, class_names):
    p = precision_score(y_true, y_pred, average=None, labels=range(len(class_names)), zero_division=0)
    r = recall_score(y_true, y_pred, average=None, labels=range(len(class_names)), zero_division=0)
    f = f1_score(y_true, y_pred, average=None, labels=range(len(class_names)), zero_division=0)
    return pd.DataFrame({"class": class_names, "precision": p, "recall": r, "f1": f})

# Representative run for each of the two headline models: 100% labels, first seed.
per_class_frames = {}
for m in ["DINOv2-frozen+MLP", "ResNet18-scratch"]:
    rec = next(r for r in records if r["model"] == m and r["fraction"] == 1.0 and r["seed"] == SEEDS[0])
    per_class_frames[m] = per_class_table(rec["y_true"], rec["preds"], CLASS_NAMES)
    print(f"\n=== {m} (100% labels, seed={SEEDS[0]}) ===")
    print(per_class_frames[m].round(3).to_string(index=False))

## Step 11 — Confusion Matrices

**Why?**
Per-class scores say *how well* each class is handled, but not *which* other class it is mixed up with. The confusion matrix shows this directly.

**What?**
Two heatmaps (one per headline model, same run as in Step 10). Rows are the true classes, columns are the predicted classes, and each cell counts galaxies.

**How?**
`confusion_matrix()` builds the table and `seaborn.heatmap` draws it with class names on both axes.

> **In simple words:** Numbers on the diagonal are correct predictions. Any large number away from the diagonal shows a pair of classes that the model finds hard to tell apart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, m in zip(axes, ["DINOv2-frozen+MLP", "ResNet18-scratch"]):
    rec = next(r for r in records if r["model"] == m and r["fraction"] == 1.0 and r["seed"] == SEEDS[0])
    cm = confusion_matrix(rec["y_true"], rec["preds"], labels=range(N_CLASSES))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(m); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

## Step 12 — Error Analysis

**Why?**
Metrics tell us *how much* error there is; error analysis tells us *what kind* of error it is. This is the qualitative complement to the numbers above.

**What?**
For each headline model, the three classes with the lowest F1 and the three most frequent confusions (true class → predicted class).

**How?**
`error_analysis()` sorts the classes by F1 and picks the largest off-diagonal entries of the confusion matrix.

> **What to look for:** Some confusions are physically natural. Round, in-between and cigar-shaped smooth galaxies form a continuous range of ellipticity, so neighbouring classes may blur into each other. Compare the printed pairs with this expectation and comment on any surprises.

In [ ]:
def error_analysis(y_true, y_pred, class_names, top_k=3):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    f1s = f1_score(y_true, y_pred, average=None, labels=range(len(class_names)), zero_division=0)
    worst_classes = np.argsort(f1s)[:top_k]
    print("Lowest-F1 classes:")
    for c in worst_classes:
        print(f"  {class_names[c]}: F1={f1s[c]:.3f}")

    off_diag = cm.copy().astype(float)
    np.fill_diagonal(off_diag, 0)
    flat_idx = np.dstack(np.unravel_index(np.argsort(-off_diag.ravel()), off_diag.shape))[0]
    print("\nMost-confused class pairs (true -> predicted, count):")
    shown = 0
    for i, j in flat_idx:
        if off_diag[i, j] <= 0:
            continue
        print(f"  {class_names[i]} -> {class_names[j]}: {int(off_diag[i, j])}")
        shown += 1
        if shown >= top_k:
            break

for m in ["DINOv2-frozen+MLP", "ResNet18-scratch"]:
    rec = next(r for r in records if r["model"] == m and r["fraction"] == 1.0 and r["seed"] == SEEDS[0])
    print(f"\n=== Error analysis: {m} ===")
    error_analysis(rec["y_true"], rec["preds"], CLASS_NAMES)

## Step 13 — t-SNE of Frozen DINOv2 Embeddings

**Why?**
If DINOv2 has learnt useful features without labels, galaxies of the same class should end up close together in embedding space even though the model never saw a label.

**What?**
A 2-D scatter plot of the test-set embeddings coloured by true class. (A matching panel for the fine-tuned model is optional and is left out by default; see the note printed by the cell.)

**How?**
`TSNE(n_components=2, perplexity=30)` reduces the 384-number embeddings to two coordinates; points are coloured using the true labels.

> **In simple words:** t-SNE draws a map in which similar galaxies are placed near each other. Clearly separated coloured groups suggest that the features already capture morphology. Treat it as a picture for intuition only: distances between clusters and cluster sizes in a t-SNE plot should not be read too literally.

In [ ]:
tsne = TSNE(n_components=2, random_state=0, perplexity=30)
emb_2d_frozen = tsne.fit_transform(test_feats)

plt.figure(figsize=(7, 6))
sc = plt.scatter(emb_2d_frozen[:, 0], emb_2d_frozen[:, 1], c=test_labels, cmap="tab10", s=8, alpha=0.7)
plt.legend(handles=sc.legend_elements()[0], labels=CLASS_NAMES, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.title("t-SNE of frozen DINOv2 embeddings (test set)")
plt.tight_layout(); plt.savefig("tsne_dinov2_frozen.png", dpi=150); plt.show()

# Optional: the fine-tuned model's embeddings can be projected the same way (see the note printed below).
ft_rec = next((r for r in records if r["model"] == "DINOv2-finetuned" and r["fraction"] == 1.0), None)
if ft_rec is not None:
    print("A fine-tuned DINOv2 model was trained at fraction=1.0. To see its t-SNE panel as well, "
          "re-extract its embeddings by passing test_ds through the fine-tuned backbone; this is "
          "left out by default to avoid an extra forward pass over the whole test set.")

## Step 14 — Attention Visualisation (DINOv2)

**Why?**
We would like to check that the model looks at the galaxy itself and not at the background — a basic test of whether its features are meaningful.

**What?**
For one test image, the attention maps of the first six heads of the **last transformer block of the DINOv2 backbone** used in this notebook (not the older DINO ViT-S/8 model).

**How?**
- A forward hook on the block's `qkv` layer captures the query / key / value projections.
- The softmax attention is recomputed manually as *softmax(Q Kᵀ ÷ √d)*.
- The row for the `[CLS]` token (the token that summarises the image) is reshaped to the 16 × 16 patch grid and enlarged to 224 × 224.

> **In simple words:** An *attention map* highlights the parts of the picture that the model weighs most when forming its summary. Different heads often specialise in different things, so their maps differ. This is a qualitative check on a single image, not a proof.

In [ ]:
def extract_dinov2_attention(model, img_tensor, block_idx=-1, patch_size=14):
    block = model.blocks[block_idx]
    captured = {}
    def hook(module, inp, out):
        captured["qkv"] = out
    # A hook is a small function that grabs the 'qkv' (query-key-value) output of the last block
    # while the image passes through the model.
    h = block.attn.qkv.register_forward_hook(hook)
    with torch.no_grad():
        _ = model(img_tensor.unsqueeze(0).to(DEVICE))
    h.remove()

    qkv = captured["qkv"]                      # (1, N, 3*dim)
    B, N, three_c = qkv.shape
    num_heads = block.attn.num_heads
    head_dim = (three_c // 3) // num_heads
    qkv = qkv.reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
    q, k, _ = qkv[0], qkv[1], qkv[2]
    # Attention = softmax(Q K^T / sqrt(d)); row 0 belongs to the [CLS] token, which summarises the image.
    attn = (q @ k.transpose(-2, -1)) * (head_dim ** -0.5)
    attn = attn.softmax(dim=-1)[0]              # (num_heads, N, N)

    n_patches = N - 1
    grid = int(round(math.sqrt(n_patches)))
    cls_attn = attn[:, 0, 1:].reshape(num_heads, grid, grid)
    cls_attn = F.interpolate(cls_attn.unsqueeze(0), scale_factor=patch_size, mode="nearest")[0]
    return cls_attn.cpu().numpy()

sample_img, sample_label = test_ds[0]
attn_maps = extract_dinov2_attention(dinov2_backbone, sample_img)

n_heads_to_show = min(6, attn_maps.shape[0])
fig, axes = plt.subplots(1, n_heads_to_show + 1, figsize=(3 * (n_heads_to_show + 1), 3))
img_disp = sample_img.permute(1, 2, 0).numpy()
img_disp = (img_disp * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)).clip(0, 1)
axes[0].imshow(img_disp); axes[0].set_title("Input"); axes[0].axis("off")
for i in range(n_heads_to_show):
    axes[i + 1].imshow(attn_maps[i], cmap="inferno")
    axes[i + 1].set_title(f"Head {i}"); axes[i + 1].axis("off")
plt.tight_layout(); plt.savefig("attention_maps_dinov2.png", dpi=150); plt.show()

## Step 15 — Computational Efficiency Comparison

**Why?**
In practice accuracy is not the only concern. A slightly less accurate model that trains in seconds and needs little memory may be the better choice for a large survey.

**What?**
For each model: number of trainable parameters, mean training time, mean inference time per image and mean peak GPU memory.

**How?**
The efficiency measurements stored in `per_run_df` are grouped by model and averaged.

**How to read this table fairly**
- **Probes count only their own small head.** For frozen-DINOv2 models the table lists the probe's trainable parameters (1,925 for the linear probe and 99,845 for the MLP probe) and times only the probe. The backbone's roughly 21 million frozen parameters and the one-time embedding extraction (`dino_extract_time`, Step 6b) are *not* included.
- **Inference time is not like-for-like.** Probes are timed on ready-made embeddings, whereas CNN and fine-tuned-DINOv2 timings include the full image pass through the network.
- **Averages mix different fractions.** Models 4 and 5 were run at only three fractions, and training time grows with the number of labelled images. Filter to a single fraction (for example 100%) before comparing training times.
- **Memory:** peak GPU memory was recorded only for the models trained end to end on the GPU (models 3, 4 and 5); missing values are shown as 0 and simply mean "not measured".

In [ ]:
# NOTE: probes are timed on pre-computed embeddings only; see the notes in Step 15 before comparing.
eff = per_run_df.groupby("model").agg(
    mean_train_time_s=("train_time", "mean"),
    mean_infer_time_per_img_ms=("infer_time", lambda x: x.mean() * 1000),
    n_params=("n_params", "first"),
    mean_peak_mem_mb=("peak_mem_mb", "mean"),
).reset_index()
eff["mean_peak_mem_mb"] = eff["mean_peak_mem_mb"].fillna(0)
eff

## Step 16 — Ablation Studies

**Why?**
An ablation removes or changes *one* ingredient at a time, so that we can say which design choice is responsible for a gain.

**What?**
- **(a) Frozen vs. fine-tuned DINOv2** — the effect of unfreezing the last transformer blocks.
- **(b) MLP probe vs. linear probe** — the effect of probe capacity and non-linearity, with the representation kept fixed.

**How?**
Two side-by-side label-efficiency plots (accuracy with 95% CI against label fraction), built from the same `agg` table.

**Caution for (a):** the fine-tuned model is also trained with data augmentation, whereas the probes use pre-computed, non-augmented embeddings, and the fine-tuned model was run at three fractions only. The comparison therefore bundles a few differences together; interpret it as "frozen probe vs. fine-tuned model" rather than a perfectly isolated effect of unfreezing.

In [ ]:
ablation_a = agg[agg.model.isin(["DINOv2-frozen+MLP", "DINOv2-finetuned"])]
ablation_b = agg[agg.model.isin(["DINOv2-frozen+MLP", "DINOv2-frozen+Linear"])]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, abl, title in [(axes[0], ablation_a, "Frozen vs. fine-tuned DINOv2"),
                        (axes[1], ablation_b, "MLP probe vs. linear probe")]:
    for m, g in abl.groupby("model"):
        g = g.sort_values("fraction")
        ax.errorbar(g["fraction"], g["acc_mean"], yerr=g["acc_ci95"], marker="o", capsize=3, label=m)
    ax.set_xlabel("Label fraction"); ax.set_ylabel("Test accuracy"); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("ablation_studies.png", dpi=150); plt.show()

## Step 17 — Comparison with AstroDINO's Reported Numbers

**Why?**
Placing our results next to the source paper's shows whether our findings point in the same direction.

**What?**
A side-by-side view of AstroDINO's reported Top-1 accuracies and our own accuracies at 100% labels.

**How?**
The table below is quoted from the paper; the code cell prints our `agg` results for the largest label fraction.

**Important caveat.** AstroDINO reports Top-1 accuracy for DINOv2 ViT-L/14 and ViT-S/14 on a **10-class** Galaxy Zoo task derived differently from ours (their labels come from an `argmax` over the MultimodalUniverse GZ2 10-column encoding, evaluated on DESI Legacy Survey cut-outs). Our task uses a different, independently documented 5-class scheme with an explicit reliability cut, so the numbers are **not directly comparable**. They are included as context, not as a benchmark to beat.

| Model (AstroDINO, 10-class task) | Reported Top-1 Accuracy |
|---|---|
| DINOv2 ViT-L/14 | 73% |
| DINOv2 ViT-S/14 | 67% |
| EfficientNetB0 (no pretraining) | 81% |
| ResNet-18 (no pretraining) | 73% |
| DenseNet121 (no pretraining) | 74% |

Our own results at 100% labels (Step 8) are printed below for side-by-side reference.

In [ ]:
print(agg[agg.fraction == agg.fraction.max()][["model", "acc_mean", "acc_std", "n_runs"]]
      .to_string(index=False))

## Step 18 — Auto-Generated Results Summary

**Why?**
Writing the discussion by hand risks mismatches between the text and the actual numbers. Generating the sentences from the computed tables avoids this.

**What?**
Short narrative statements: accuracy of frozen DINOv2 + MLP against ResNet-18 from scratch at full and at the lowest label fraction, the gap between them in percentage points, and whether the low-label difference is statistically significant.

**How?**
The cell reads the `agg` and `sig_df` tables, formats the values as percentages and prints the sentences. Run it last and copy its output into the paper's Discussion section.

In [ ]:
def fmt_pct(x): return f"{100*x:.1f}%"      # e.g. 0.731 -> "73.1%"
def fmt_pp(x): return f"{100*x:.1f}"         # e.g. 0.123 -> "12.3" (used before the words "percentage points")

best_frac = agg["fraction"].max()
worst_frac = agg["fraction"].min()

def get(model, frac, col):
    row = agg[(agg.model == model) & (agg.fraction == frac)]
    return row[col].values[0] if len(row) else float("nan")

summary_lines = []
summary_lines.append(
    f"At full label availability ({fmt_pct(best_frac)} of the training set), the frozen "
    f"DINOv2 + MLP probe reached {fmt_pct(get('DINOv2-frozen+MLP', best_frac, 'acc_mean'))} "
    f"(± {get('DINOv2-frozen+MLP', best_frac, 'acc_std'):.3f} std over {SEEDS.__len__()} seeds) "
    f"test accuracy, versus {fmt_pct(get('ResNet18-scratch', best_frac, 'acc_mean'))} for the "
    f"from-scratch ResNet-18."
)
summary_lines.append(
    f"At the lowest label fraction tested ({fmt_pct(worst_frac)}), frozen DINOv2 + MLP reached "
    f"{fmt_pct(get('DINOv2-frozen+MLP', worst_frac, 'acc_mean'))} versus "
    f"{fmt_pct(get('ResNet18-scratch', worst_frac, 'acc_mean'))} for the from-scratch CNN — a gap of "
    f"{fmt_pp(get('DINOv2-frozen+MLP', worst_frac, 'acc_mean') - get('ResNet18-scratch', worst_frac, 'acc_mean'))} "
    f"percentage points, versus a gap of "
    f"{fmt_pp(get('DINOv2-frozen+MLP', best_frac, 'acc_mean') - get('ResNet18-scratch', best_frac, 'acc_mean'))} "
    f"percentage points at full labels."
)
sig_row = sig_df[(sig_df.model_a == "DINOv2-frozen+MLP") & (sig_df.model_b == "ResNet18-scratch")
                  & (sig_df.fraction == worst_frac)]
if len(sig_row):
    p = sig_row["t_p"].values[0]
    summary_lines.append(
        f"This difference at {fmt_pct(worst_frac)} labels is "
        f"{'statistically significant (p = %.4f < 0.05, paired t-test)' % p if p < 0.05 else 'not statistically significant (p = %.4f, paired t-test)' % p}."
    )

print("\n".join(summary_lines))

## Limitations and Future Work

**Limitations**
- **Single survey and small subset.** All results come from one dataset (GZ2), a 5-class scheme and a class-balanced subset of at most 20,000 galaxies.
- **Only the smallest DINOv2.** ViT-S/14 is used; larger backbones (ViT-B, ViT-L, ViT-g) are not tested.
- **Clean-sample bias.** Ambiguous galaxies are dropped, so the reported accuracy applies to confidently labelled galaxies and will look better than performance on *all* galaxies. Volunteer labels themselves also contain noise.
- **No hyperparameter tuning.** Learning rates and epoch counts are fixed; the validation set is not used for model selection.
- **Reduced grid for two models.** ResNet-18 (ImageNet fine-tuned) and the fine-tuned DINOv2 run at only three label fractions, and they are not part of the significance tests in Step 9.
- **Few seeds.** With 5 seeds the confidence intervals are approximate and the Wilcoxon test has limited power.
- **Timing noise.** Training and inference times measured on shared Colab GPUs vary between sessions.

**Future work**
- Extend the significance tests to the fine-tuned and ImageNet-pretrained models on the full fraction grid.
- Add the t-SNE panel of the fine-tuned model and a k-nearest-neighbour probe.
- Tune hyperparameters on the validation set and test larger DINOv2 backbones.
- Repeat the study on other galaxy surveys (for example DECaLS or DESI) to check how well the conclusions carry over.

## References

1. M. Caron et al., "Emerging Properties in Self-Supervised Vision Transformers," arXiv:2104.14294, 2021.
2. M. Oquab et al., "DINOv2: Learning Robust Visual Features without Supervision," arXiv:2304.07193, 2023.
3. V. Manwadkar and S. Kapare, "AstroDINO: Self-Supervised Learning on Astronomical Images," Stanford University, 2025. *(Source paper — pipeline reproduced and extended here.)*
4. B. Willett et al., "Galaxy Zoo 2: detailed morphological classifications for 304,122 galaxies from the Sloan Digital Sky Survey," MNRAS, 435(3):2835–2860, 2013.
5. R. Hart et al., "Galaxy Zoo: comparing the demographics of spiral arm number and a new method for correcting redshift bias," MNRAS, 461(4):3663–3682, 2016.
6. A. Dey et al., "Overview of the DESI Legacy Imaging Surveys," AJ, 157(5):168, 2019.
7. The Multimodal Universe Collaboration et al., "The Multimodal Universe," arXiv:2412.02527, 2024.
8. M. Walmsley et al., "Galaxy Zoo DECaLS: Detailed visual morphology measurements," MNRAS, 509(3):3966–3988, 2022 (galaxy-datasets / Zoobot package).
9. K. He et al., "Deep Residual Learning for Image Recognition," CVPR, 2016.